# Running K-mouflage: the essentials

How to build a model, run the solver, check it, calibrate it, compare it
to another model, and save the result. 

In [1]:
import sys
sys.path.insert(0, "..")

from kmouflage import (
    CosmologicalParams, InitialConditions,
    make_powerlaw_K, make_arctan_K, make_exponential_coupling, make_exponential_potential,
    run_background, verify, calibrate_M4_tilde,
    summary_table, print_summary_table, make_lcdm_reference,
    KmouflageGrowth, LCDMGrowth,
    save_table,
)

## 1. Cosmology & initial conditions

In [2]:
cosmo = CosmologicalParams(
    omega_m=0.1430,       # Omega_m0 * h^2 (Planck 2018)
    omega_r=4.15e-5,      # Omega_r0 * h^2 (Planck 2018)
    H100=100.0,           # fixed reference unit [km/s/Mpc]
    M_Pl_input=2.435e18,  # reduced Planck mass [GeV]
    M4_tilde=1.0,         # free parameter, set it here instead of calibrating (section 6)
)
ic = InitialConditions(
    z_ini=1e5,            # redshift where the integration starts (deep RD)
    phi_ini=0.0,          # initial scalar field value
)

print(cosmo)
print(ic)

CosmologicalParams(H100=100.0, Mpl=1.0, M_Pl_input=2.435e+18, omega_m=0.143, omega_r=4.15e-05, M4_tilde=1.0)
InitialConditions(z_ini=100000.0, phi_ini=0.0)


## 2. Choose K(X), the coupling A(φ), and the potential f(φ)

model = K(X) kinetic term, coupling = A(φ) conformal coupling to matter, potential = f(φ). Leaving potential=None (the default) is bare K-mouflage (f≡1), adding one, like below, is what this project calls a Kinessence model.

All three are built with make_* factories, see `custom_models_guide.ipynb` to write your own instead.

In [3]:
model     = make_powerlaw_K(K0=1, m=3)             # K(X) = X + K0*X^m
coupling  = make_exponential_coupling(beta=0.1)    # A(phi) = exp(beta*phi)
potential = make_exponential_potential(lam=1.0)    # f(phi) = exp(-lam*phi) -> Kinessence

## 3. Run it

run_background builds a KMouflageBackground and calls bg.run(). The object then exposes every derived quantity as a function of N = ln(a) (bg.phi(N), bg.Omega_m(N), ...). Passing potential=potential is what makes this a Kinessence run instead of bare K-mouflage.

In [4]:
bg = run_background(model, coupling, potential=potential, cosmo=cosmo, ic=ic, verbose=True)

print(f"H0_predicted        = {bg.H0_predicted:.4f} km/s/Mpc")
print(f"Omega_m0_predicted  = {bg.Omega_m0_predicted:.4f}")
print(f"Omega_de0_predicted = {bg.Omega_de0_predicted:.4f}")
print(f"phi(z=0)            = {bg.phi(0.0):.4f}")


[RUN] z_ini=1.0e+05 | omega_m=0.143 | omega_r=4.15e-05


[OK] φ̃(z=0)=-4.8587e-02 | M_Pl0_eff=1.004871 | delta_Mpl=+4.8705e-03
[PREDICTED] h=0.765908 | H0=76.5908 km/s/Mpc | Omega_m0=0.2438 | Omega_de0=0.7562
[CHECK] Omega_DE(z_ini=1.0e+05) = -4.151e-08 (should be << 1 for the deep-RD omega_m/omega_r extrapolation to be valid)
H0_predicted        = 76.5908 km/s/Mpc
Omega_m0_predicted  = 0.2438
Omega_de0_predicted = 0.7562
phi(z=0)            = -0.0486


## 3bis. Any physical quantity, bg.<name>(N)

In [5]:
N = 0.0   # N = ln(a); any N, or a numpy array of N, works

print("rho_m         =", bg.rho_m(N))          # matter density
print("rho_r         =", bg.rho_r(N))          # radiation density
print("rho_phi       =", bg.rho_phi(N))        # scalar field density
print("p_phi         =", bg.p_phi(N))          # scalar field pressure
print("w_phi         =", bg.w_phi(N))          # scalar field equation of state
print("w_de_def      =", bg.w_de_def(N))       # dark energy equation of state
print("mu_K          =", bg.mu_K(N))           # effective Newton constant
print("epsilon_F     =", bg.epsilon_F(N))      # BBN / G_eff constraint quantity
print("Z             =", bg.Z(N))              # kinetic normalization
print("Z_eff         =", bg.Z_eff(N))          # kinetic normalization, no-ghost condition
print("F             =", bg.F(N))              # (M_Pl,eff / M_Pl)^2
print("H             =", bg.H(N))              # physical Hubble rate
print("t_cosmic      =", bg.t_cosmic(N))       # cosmic time

# full list: see KMouflageBackground._build_interpolators in solver.py

rho_m         = 0.42899999999999994
rho_r         = 0.0001245
rho_phi       = 1.1553083278136667
p_phi         = -0.9880001274442325
w_phi         = -0.8551830742135719
w_de_def      = -0.8425698071613772
mu_K          = 1.0109223217621122
epsilon_F     = -0.009670328262275762
Z             = 0.9712203128938611
Z_eff         = 1.0593860794103194
F             = 1.009764756664624
H             = 0.765907866150116
t_cosmic      = 1.337963040511194


## 4. Sanity-check, verify

In [6]:
verify(bg)

[verify] work in progress, no checks implemented yet.


True

## 5. Comparing models


In [7]:
bg_arctan = run_background(make_arctan_K(K_star=1000, X_star=100), coupling, potential=potential, cosmo=cosmo, ic=ic)
bg_lcdm   = run_background(model, make_exponential_coupling(beta=0.0), potential=potential, cosmo=cosmo, ic=ic)  # beta=0 -> A=1

rows = summary_table(
    {"power-law": bg, "arctan": bg_arctan, "ΛCDM-like (β=0)": bg_lcdm},
    ref=make_lcdm_reference(cosmo),
)
print_summary_table(rows)

model                    H0    ΔH0(%)       Ω_m0   ΔΩ_m0(%)      Ω_DE0   ΔΩ_DE0(%)      w_DE0   Δw_DE0(%)
─────────────────────────────────────────────────────────────────────────────────────────────────────────
power-law           76.5908   +13.704    0.24377    -22.652    0.75616     +10.429   -0.84257     -15.743
arctan              79.3899   +17.859    0.22688    -28.010    0.77305     +12.895   -0.82928     -17.072
ΛCDM-like (β=0)     64.3057    -4.534    0.34581     +9.725    0.65409      -4.477   -0.86051     -13.949


/home/tomcana/document/stage/Kmouflage_project/examples/../kmouflage/solver.py:247: RuntimeWarning: divide by zero encountered in divide
  p_de_arr / rho_de_fried_arr,  -1.0)


## 6. Calibrating M4_tilde to a target H0

M4_tilde (the kinetic term's amplitude) is the one free parameter left after choosing a model. calibrate_M4_tilde tunes it so bg.H0_predicted matches a target, mutating bg in place and re-running it.

In [8]:
# bg.M4_tilde = 0.9; bg.run(verbose=False)   # skip calibration, set M4_tilde directly instead
M4_tilde, H0_final, n_iter = calibrate_M4_tilde(bg, target_H0=67.36, verbose=False)
print(f"M4_tilde = {M4_tilde:.6f}  ->  H0_predicted = {bg.H0_predicted:.4f} km/s/Mpc  ({n_iter} iterations)")

M4_tilde = 0.670180  ->  H0_predicted = 67.3600 km/s/Mpc  (10 iterations)


## 7. Linear growth

KmouflageGrowth(bg) derives the growth rate f = dlnD+/dN and growth factor D+ from an already-run background; LCDMGrowth is the independent closed-form ΛCDM version.

In [9]:
growth      = KmouflageGrowth(bg, N_points=2000).run(verbose=False)
growth_lcdm = LCDMGrowth(Omega_m0=bg.Omega_m0_predicted, Omega_r0=bg.Omega_r0_predicted, N_points=2000).run(verbose=False)

print(f"f(z=0)  : model={growth['f'][-1]:.5f}   ΛCDM={growth_lcdm['f'][-1]:.5f}")
print(f"D+(z=0) : model={growth['Dp'][-1]:.5f}   ΛCDM={growth_lcdm['Dp'][-1]:.5f}")

f(z=0)  : model=0.55928   ΛCDM=0.52723
D+(z=0) : model=1.00000   ΛCDM=1.00000


## 8. Saving the result

save_table writes one CSV: a commented header with the run's model/coupling/potential/cosmology, then the chosen fields as columns. Read it back.

In [10]:
from kmouflage.run import DEFAULT_FIELDS
print(DEFAULT_FIELDS)   # saved by default if fields= is omitted

path = save_table(bg, outdir="runs")
# path = save_table(bg, outdir="runs", fields=["Z", "mu_K", "rho_phi", "p_phi"])   # only the fields you want
print(path)

with open(path) as f:
    print("".join(f.readlines()[:7]))

['z', 'a', 'phi', 'phi_prime', 't_cosmic', 'eta', 't_superconform', 'H', 'H_conf', 'Omega_m', 'Omega_r', 'Omega_de_def', 'w_de_def', 'w_phi']


runs/kmouflage_table_exponential_beta-0.1_power_law_K0-1_m-3_exponential_lam-1.0.csv
# model: power-law (K0=1, m=3)
# coupling: exponential (β=0.1)
# potential: exponential (lam=1.0)
# omega_m=0.143 omega_r=4.15e-05 H100=100.0 M4_tilde=0.6701799914911576
# z_ini=100000.0 phi_ini=0.0
# H0_predicted=67.35999999999758 Omega_m0_predicted=0.3151605723280957 Omega_de0_predicted=0.6847479649883265
N,a,phi,phi_prime,z,t_cosmic,eta,t_superconform,H,H_conf,Omega_m,Omega_r,Omega_de_def,w_de_def,w_phi

